# HeatUp — Notebook 4: Literature Validation

**LGPS is the key test:** +25 meV/atom at 0 K, should cross onto hull ~600–800 K.

| Material | MP ID | Test |
|----------|-------|------|
| LGPS | mp-696138 | Non-trivial T-hull crossover |
| Li₃PS₄ β | mp-985591 | σ=0.2 mS/cm; on hull at 0 K |
| AgI β | mp-22925 | Gate 1+2 elastic & VDOS |
| Li₂O | mp-1960 | B=87 GPa elastic reference |

**Competing phases for LGPS hull:** mp-985591, mp-3435, mp-2286, mp-2055, mp-557592.

**Edit only Configuration.**

In [ ]:
import json,os,shutil,warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import heatup.config as cfg
from heatup import run_stability_pipeline
from heatup.md_utils import load_analysis,print_database_summary,TIMESTEP_FS,N_STEPS,NBLOCK
from heatup.structure_utils import run_relaxation_subprocess,run_phonons_subprocess,run_elastic_subprocess,run_md_subprocess,prepare_aimd_folders
from heatup.anharmonic_phonons import compute_anharmonic_phonons_for_sim
warnings.filterwarnings('ignore')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Configuration

In [ ]:
cfg.MACE_MODEL=os.environ.get('MACE_MODEL_PATH','mace-mpa-0-medium')
MP_API_KEY=os.environ.get('MP_API_KEY','YOUR_MP_API_KEY_HERE')
DATABASE_ROOT='database'
CANDIDATES_DIR='input/candidates'
HULL_TEMPERATURES=list(range(0,1501,50))
OPERATING_T=800.0  # LGPS synthesis temperature
RUN_DOWNLOAD=True; RUN_STRUCTURE=True; RUN_MD=True
RUN_ANHARMONIC=True; RUN_STABILITY=True; FORCE_RERUN=False
os.makedirs(DATABASE_ROOT,exist_ok=True); os.makedirs(CANDIDATES_DIR,exist_ok=True)
print(f'MACE: {cfg.MACE_MODEL}')

## Validation systems

In [ ]:
VALIDATION_SYSTEMS=[
    {'mp_id':'mp-696138','material':'LGPS','symmetry':'P42-nmc','temperatures':[600,900],
     'ref':{'B':None,'G':None,'hull_0K_meV':25.0,'hull_stable_above_K':600,'sigma':12.0,'sigma_ref':'Kamaya Nat.Mater.10(2011)'}},
    {'mp_id':'mp-985591','material':'Li3PS4','symmetry':'Pmn2_1','temperatures':[300,600],
     'ref':{'B':25.0,'G':14.0,'hull_0K_meV':0.0,'hull_stable_above_K':0,'sigma':0.2,'sigma_ref':'Tachez SSI 14(1984)'}},
    {'mp_id':'mp-22925','material':'AgI','symmetry':'P6_3mc','temperatures':[300,600],
     'ref':{'B':35.0,'G':14.0,'hull_0K_meV':0.0,'hull_stable_above_K':0,'sigma':None,'sigma_ref':'Guerel PRB74(2006)'}},
    {'mp_id':'mp-1960','material':'Li2O','symmetry':'Fm-3m','temperatures':[300],
     'ref':{'B':87.0,'G':58.0,'hull_0K_meV':0.0,'hull_stable_above_K':0,'sigma':None,'sigma_ref':'Shi JAC456(2008)'}},
]
LGPS_COMPETING={'mp-985591':('Li3PS4','Pmn2_1'),'mp-3435':('Li4GeS4','Pmma'),
                'mp-2286':('Li2S','Fm-3m'),'mp-2055':('GeS2','P3121'),'mp-557592':('P2S5','P1')}
print(f'{len(VALIDATION_SYSTEMS)} systems defined.')
for s in VALIDATION_SYSTEMS:
    print(f"  {s['material']:<10} {s['mp_id']}  hull_0K={s['ref']['hull_0K_meV']} meV")

## Download POSCARs from Materials Project

In [ ]:
if RUN_DOWNLOAD:
    if MP_API_KEY=='YOUR_MP_API_KEY_HERE':
        raise ValueError('Set MP_API_KEY. Get yours at https://next-gen.materialsproject.org/api')
    from mp_api.client import MPRester
    from pymatgen.io.vasp import Poscar
    all_ids=list(dict.fromkeys([s['mp_id'] for s in VALIDATION_SYSTEMS]+list(LGPS_COMPETING)))
    print(f'Querying MP for {len(all_ids)} structures...')
    with MPRester(MP_API_KEY) as mpr:
        docs=mpr.materials.summary.search(material_ids=all_ids,
            fields=['material_id','formula_pretty','structure','symmetry','energy_per_atom','band_gap'],
            all_fields=False)
    docs_by_id={str(d.material_id):d for d in docs}
    print(f'Received {len(docs_by_id)} documents.')
    for s in VALIDATION_SYSTEMS:
        doc=docs_by_id.get(s['mp_id'])
        if not doc or not doc.structure: print(f"  [MISS] {s['mp_id']}"); continue
        sd=os.path.join(DATABASE_ROOT,s['material'],s['symmetry'])
        os.makedirs(sd,exist_ok=True)
        dst=os.path.join(sd,'POSCAR')
        if not os.path.exists(dst) or FORCE_RERUN:
            p=Poscar(doc.structure); p.comment=f"{doc.formula_pretty} ({s['mp_id']})"; p.write_file(dst)
            json.dump({'material_id':s['mp_id'],'formula':doc.formula_pretty,'symmetry':s['symmetry'],
                       'energy_per_atom':doc.energy_per_atom,'band_gap':doc.band_gap},
                      open(os.path.join(sd,'metadata.json'),'w'),indent=2)
            print(f"  {s['material']}/{s['symmetry']} -> {dst}")
    for mpid,(formula,sg) in LGPS_COMPETING.items():
        doc=docs_by_id.get(mpid)
        if not doc or not doc.structure: continue
        cd=os.path.join(CANDIDATES_DIR,formula,sg); os.makedirs(cd,exist_ok=True)
        dst=os.path.join(cd,'POSCAR')
        if not os.path.exists(dst) or FORCE_RERUN:
            Poscar(doc.structure).write_file(dst)
            json.dump({'material_id':mpid,'formula':formula,'symmetry':sg,
                       'energy_per_atom':doc.energy_per_atom,'band_gap':doc.band_gap},
                      open(os.path.join(cd,'metadata.json'),'w'),indent=2)
            print(f'  {formula}/{sg} ({mpid}) -> candidates/')
    print('Download complete.')
else:
    print('RUN_DOWNLOAD=False')

## Structure prep + AIMD + anharmonic VDOS

In [ ]:
for s in VALIDATION_SYSTEMS:
    sd=os.path.join(DATABASE_ROOT,s['material'],s['symmetry'])
    tag=f"{s['material']}/{s['symmetry']}"
    if not os.path.exists(os.path.join(sd,'POSCAR')): print(f'[SKIP] {tag}: POSCAR missing'); continue
    print(f'\n{"="*55}\n  {tag}\n{"="*55}')
    if RUN_STRUCTURE:
        run_relaxation_subprocess(sd,device=DEVICE,force_rerun=FORCE_RERUN)
        run_phonons_subprocess(sd,device=DEVICE,force_rerun=FORCE_RERUN)
        run_elastic_subprocess(sd,device=DEVICE,force_rerun=FORCE_RERUN)
    if RUN_MD and os.path.exists(os.path.join(sd,'relaxation','CONTCAR')):
        try: prepare_aimd_folders(sd,temperatures=s['temperatures'],force_rebuild=FORCE_RERUN)
        except Exception as exc: print(f'  [WARN] AIMD prep: {exc}'); continue
        for T in s['temperatures']:
            if run_md_subprocess(sd,T,device=DEVICE,force_rerun=FORCE_RERUN) and RUN_ANHARMONIC:
                try: compute_anharmonic_phonons_for_sim(os.path.join(sd,'aimd',f'{T}K'),temperatures=HULL_TEMPERATURES,force_recompute=FORCE_RERUN)
                except Exception as exc: print(f'  [WARN] anh: {exc}')

## HeatUp stability assessment

In [ ]:
reports={}
if RUN_STABILITY:
    for s in VALIDATION_SYSTEMS:
        sd=os.path.join(DATABASE_ROOT,s['material'],s['symmetry'])
        tag=f"{s['material']}/{s['symmetry']}"
        if not os.path.isdir(sd): continue
        print(f'\n  HeatUp: {tag}')
        rpt=run_stability_pipeline(sym_dir=sd,operating_T=OPERATING_T,
            candidates_root=CANDIDATES_DIR,database_root=DATABASE_ROOT,
            temperatures=HULL_TEMPERATURES,device=DEVICE,
            generate_missing_phases=True,force_rerun=FORCE_RERUN,save_figure=True)
        reports[tag]=rpt
        print(f'  Overall: {rpt["overall"].upper()}')

## KEY RESULT — LGPS hull crossover

In [ ]:
lgps=reports.get('LGPS/P42-nmc',{}).get('thermodynamic',{})
if lgps.get('hull_results'):
    valid=[(r['T'],r['e_above_hull_eV_atom']) for r in lgps['hull_results'] if r.get('e_above_hull_eV_atom') is not None]
    T_a=np.array([v[0] for v in valid]); E_m=np.array([v[1] for v in valid])*1000
    cross=T_a[np.argmin(np.abs(E_m))]; E_cross=float(np.interp(cross,T_a,E_m))
    fig,ax=plt.subplots(figsize=(8,4))
    ax.fill_between(T_a,-50,0,color='#d5f5e3',alpha=0.5,label='Stable')
    ax.fill_between(T_a,0,100,color='#fadbd8',alpha=0.35,label='Unstable')
    ax.plot(T_a,E_m,color='#0072B2',lw=2,label='LGPS (HeatUp)')
    ax.scatter([0],[25],color='red',s=80,zorder=5,label='DFT 0K ref +25 meV (Ong 2013)')
    ax.axhline(0,color='green',lw=0.9,ls='--')
    ax.axvline(OPERATING_T,color='gray',lw=0.8,ls=':',label=f'T_op={OPERATING_T:.0f} K')
    ax.axvline(cross,color='#0072B2',lw=0.8,ls=':',label=f'Crossover~{cross:.0f} K')
    ax.set_xlabel('Temperature (K)'); ax.set_ylabel('\u0394E_hull (meV/atom)')
    ax.set_title('LGPS T-dependent stability (key validation)')
    ax.legend(frameon=False,fontsize=8); ax.set_xlim(0,1500); ax.set_ylim(-50,80)
    plt.tight_layout(); plt.savefig('lgps_hull_validation.pdf',dpi=300,bbox_inches='tight'); plt.show()
    agree='PASS' if 400<cross<1000 else 'FAIL'
    print(f'HeatUp 0K: {E_m[0]:+.1f} meV/atom  (ref: +25)  Crossover: {cross:.0f} K  -> {agree}')
else:
    print('No LGPS hull data — complete MD and stability steps.')

## Validation tables: Gate 1 + conductivity

In [ ]:
print(f'{"Material":<22}{"B_HU":>8}{"B_ref":>7}{"G_HU":>8}{"G_ref":>7}{"Born":>6}{"Agree":>6}')
print('-'*62)
for s in VALIDATION_SYSTEMS:
    tag=f"{s['material']}/{s['symmetry']}"; m=reports.get(tag,{}).get('mechanical',{}); ref=s['ref']
    B=m.get('bulk_modulus_GPa'); G=m.get('shear_modulus_GPa'); Br=ref.get('B'); Gr=ref.get('G')
    born='\u2713' if m.get('born_stable') else ('-' if m.get('born_stable') is None else '\u2717')
    agree='\u2713' if B and Br and abs(B-Br)/Br<0.20 else ('-' if not Br else '\u26a0')
    print(f"{tag:<22}{B or '-':>8}{Br or 'N/A':>7}{G or '-':>8}{Gr or 'N/A':>7}{born:>6}{agree:>6}")
print('\nModuli in GPa. \u2713 = within 20% of DFT reference.\n')
print(f'{"Material":<15}{"T":>5}{"Sp":>4}{"D (cm2/s)":>14}{"sigma_HU":>12}{"sigma_exp":>11}')
print('-'*63)
for s in VALIDATION_SYSTEMS:
    if not s['ref'].get('sigma'): continue
    for T in s['temperatures']:
        an=load_analysis(os.path.join(DATABASE_ROOT,s['material'],s['symmetry'],'aimd',f'{T}K'))
        if not an: continue
        for sp,vals in an.get('diffusion',{}).items():
            D=vals.get('diffusivity_cm2s',0); sg=vals.get('conductivity_mScm',0)
            if D<1e-15: continue
            print(f"{s['material']}/{s['symmetry']:<15}{T:>5}{sp:>4}{D:>14.3e}{sg:>12.4f}{s['ref']['sigma']:>11.2f}")

## Save report

In [ ]:
with open('validation_results.json','w') as fh:
    json.dump({'mace':cfg.MACE_MODEL,'T_op':OPERATING_T,
               'reports':{k:{ck:cv for ck,cv in v.items() if ck!='vibrational'} for k,v in reports.items()}},
              fh,indent=2,default=str)
print('Saved: validation_results.json')
for s in VALIDATION_SYSTEMS:
    tag=f"{s['material']}/{s['symmetry']}"; r=reports.get(tag,{})
    print(f"  {tag:<25} {r.get('overall','N/A').upper()}")